In [0]:
from graphframes import *
from pyspark.sql.functions import col, desc
import os

Loading Data

In [0]:
Initial_edges = spark.read.option("inferSchema", "true").option("header", "false").option("sep", " ").csv("dbfs:/FileStore/tables/facebook_combined_txt.gz").toDF("src","dst")

In [0]:
Initial_edges.show()

+---+---+
|src|dst|
+---+---+
|  0|  1|
|  0|  2|
|  0|  3|
|  0|  4|
|  0|  5|
|  0|  6|
|  0|  7|
|  0|  8|
|  0|  9|
|  0| 10|
|  0| 11|
|  0| 12|
|  0| 13|
|  0| 14|
|  0| 15|
|  0| 16|
|  0| 17|
|  0| 18|
|  0| 19|
|  0| 20|
+---+---+
only showing top 20 rows



In [0]:
Initial_edges.filter("(src = 0 AND dst = 1) OR (src = 1 AND dst = 0)").show()

+---+---+
|src|dst|
+---+---+
|  0|  1|
+---+---+



Create Graphs

In [0]:
reversed_edges = Initial_edges.select(col("dst").alias("src"), col("src").alias("dst"))
facebook_edge = Initial_edges.union(reversed_edges).distinct()

In [0]:
facebook_edge.show()

+---+---+
|src|dst|
+---+---+
|  0| 13|
|  0| 20|
|  0| 14|
|  0|  7|
|  0|  5|
|  0| 16|
|  0| 10|
|  0| 17|
|  0|  9|
|  0| 15|
|  0| 18|
|  0| 11|
|  0|  1|
|  0| 19|
|  0|  8|
|  0|  6|
|  0|  2|
|  0|  3|
|  0|  4|
|  0| 12|
+---+---+
only showing top 20 rows



In [0]:
facebook_edge.filter("(src = 0 AND dst = 1) OR (src = 1 AND dst = 0)").show()

+---+---+
|src|dst|
+---+---+
|  0|  1|
|  1|  0|
+---+---+



In [0]:
facebook_vertices = facebook_edge.select("src").union(facebook_edge.select("dst")).distinct().withColumnRenamed("src", "id")

In [0]:
facebook_vertices.show()

+---+
| id|
+---+
| 22|
|  1|
| 13|
|  6|
| 16|
|  3|
| 20|
|  5|
| 19|
|  9|
| 17|
|  4|
|  8|
| 23|
|  7|
| 10|
| 24|
| 21|
| 14|
|  2|
+---+
only showing top 20 rows



In [0]:
facebook_graph = GraphFrame(facebook_vertices, facebook_edge)

Running Queries

1. Find the top 5 nodes with the highest outdegree and find the count of the number of outgoing edges in each

In [0]:
facebook_OutDegree = facebook_graph.outDegrees.orderBy(desc("outDegree")).limit(5)

In [0]:
facebook_OutDegree.coalesce(1).write.option("header", "true").mode("overwrite").csv("dbfs:/FileStore/tables/facebook_output/top5_outdegree")

files = dbutils.fs.ls("dbfs:/FileStore/tables/facebook_output/top5_outdegree")
part_file = [file.path for file in files if "part" in file.name][0]

dbutils.fs.mv(part_file, "dbfs:/FileStore/tables/facebook_output/top5_outdegree.csv")

dbutils.fs.rm("dbfs:/FileStore/tables/facebook_output/top5_outdegree", True)

Out[73]: True

In [0]:
facebook_OutDegree.show()

+----+---------+
|  id|outDegree|
+----+---------+
| 107|     1045|
|1684|      792|
|1912|      755|
|3437|      547|
|   0|      347|
+----+---------+



2. Find the top 5 nodes with the highest indegree and find the count of the number of incoming edges in each

In [0]:
facebook_InDegree = facebook_graph.inDegrees.orderBy(desc("inDegree")).limit(5)

In [0]:
facebook_InDegree.coalesce(1).write.option("header", "true").mode("overwrite").csv("dbfs:/FileStore/tables/facebook_output/top5_indegree")

files = dbutils.fs.ls("dbfs:/FileStore/tables/facebook_output/top5_indegree")
part_file = [file.path for file in files if "part" in file.name][0]

dbutils.fs.mv(part_file, "dbfs:/FileStore/tables/facebook_output/top5_indegree.csv")

dbutils.fs.rm("dbfs:/FileStore/tables/facebook_output/top5_indegree", True)

Out[76]: True

In [0]:
facebook_InDegree.show()

+----+--------+
|  id|inDegree|
+----+--------+
| 107|    1045|
|1684|     792|
|1912|     755|
|3437|     547|
|   0|     347|
+----+--------+



3. Calculate PageRank for each of the nodes and output the top 5 nodes with the highest PageRank values. You are free to define any suitable parameters.

In [0]:
facebook_pageRank = facebook_graph.pageRank(resetProbability=0.15, tol=0.01).vertices.select("id", "pagerank").orderBy(desc("pagerank")).limit(5)

In [0]:
facebook_pageRank.coalesce(1).write.option("header", "true").mode("overwrite").csv("dbfs:/FileStore/tables/facebook_output/top5_pagerank")

files = dbutils.fs.ls("dbfs:/FileStore/tables/facebook_output/top5_pagerank")
part_file = [file.path for file in files if "part" in file.name][0]

dbutils.fs.mv(part_file, "dbfs:/FileStore/tables/facebook_output/top5_pagerank.csv")

dbutils.fs.rm("dbfs:/FileStore/tables/facebook_output/top5_pagerank", True)

Out[79]: True

In [0]:
facebook_pageRank.show()

+----+------------------+
|  id|          pagerank|
+----+------------------+
|3437|29.405770490380842|
| 107|26.855937495982364|
|1684|24.625082773933237|
|   0| 23.92001072763395|
|1912|15.135237056365433|
+----+------------------+



4. Run the connected components algorithm on it and find the top 5 components with the largest number of nodes.

In [0]:
sc.setCheckpointDir("/tmp/spark-checkpoints")

In [0]:
connected_components = facebook_graph.connectedComponents()

In [0]:
facebook_conncomp = connected_components.groupBy("component").count().orderBy("count", ascending=False).limit(5)

In [0]:
facebook_conncomp.coalesce(1).write.option("header", "true").mode("overwrite").csv("dbfs:/FileStore/tables/facebook_output/top5_conncomp")

files = dbutils.fs.ls("dbfs:/FileStore/tables/facebook_output/top5_conncomp")
part_file = [file.path for file in files if "part" in file.name][0]

dbutils.fs.mv(part_file, "dbfs:/FileStore/tables/facebook_output/top5_conncomp.csv")

dbutils.fs.rm("dbfs:/FileStore/tables/facebook_output/top5_conncomp", True)

Out[84]: True

In [0]:
facebook_conncomp.show()

+---------+-----+
|component|count|
+---------+-----+
|        0| 4039|
+---------+-----+



5. Run the triangle counts algorithm on each of the vertices and output the top 5 vertices with the largest triangle count. In case of ties, you can randomly select the top 5 vertices.

In [0]:
tricount = facebook_graph.triangleCount()
facebook_tricount = tricount.select("id", "count").orderBy(desc("count")).limit(5)

In [0]:
facebook_tricount.coalesce(1).write.option("header", "true").mode("overwrite").csv("dbfs:/FileStore/tables/facebook_output/top5_tricount")

files = dbutils.fs.ls("dbfs:/FileStore/tables/facebook_output/top5_tricount")
part_file = [file.path for file in files if "part" in file.name][0]

dbutils.fs.mv(part_file, "dbfs:/FileStore/tables/facebook_output/top5_tricount.csv")

dbutils.fs.rm("dbfs:/FileStore/tables/facebook_output/top5_tricount", True)

Out[87]: True

In [0]:
facebook_tricount.show()

+----+-----+
|  id|count|
+----+-----+
|1912|30025|
| 107|26750|
|2347|16863|
|2266|16174|
|2206|15844|
+----+-----+



In [0]:
#dbutils.fs.rm("dbfs:/FileStore/tables/facebook_output", True)

In [0]:
dbutils.fs.ls("dbfs:/FileStore/tables/facebook_output")

Out[90]: [FileInfo(path='dbfs:/FileStore/tables/facebook_output/top5_conncomp.csv', name='top5_conncomp.csv', size=23, modificationTime=1745631713000),
 FileInfo(path='dbfs:/FileStore/tables/facebook_output/top5_indegree.csv', name='top5_indegree.csv', size=54, modificationTime=1745630448000),
 FileInfo(path='dbfs:/FileStore/tables/facebook_output/top5_outdegree.csv', name='top5_outdegree.csv', size=55, modificationTime=1745630442000),
 FileInfo(path='dbfs:/FileStore/tables/facebook_output/top5_pagerank.csv', name='top5_pagerank.csv', size=127, modificationTime=1745630465000),
 FileInfo(path='dbfs:/FileStore/tables/facebook_output/top5_tricount.csv', name='top5_tricount.csv', size=63, modificationTime=1745631723000)]